# Premier League Match Prediction — Scraped External Features

This notebook extends the market-odds baseline with **externally scraped, strictly
pre-match features** and evaluates whether they push 3-class (Home / Draw / Away)
balanced accuracy toward the 65% target.

**New data sources (see `scripts/scrape/`):**

| Source | Signal | Why it might help |
|---|---|---|
| **ClubElo** (`fetch_clubelo.py`) | Pre-match Elo rating & win-expectancy | Compact cross-season team strength; fixes promoted-team / early-season cold start |
| **Understat** (`_understat_sink.py`) | Rolling **expected-goals (xG)** form | De-noised strength — xG is less luck-driven than actual goals |
| **Transfermarkt** (`fetch_transfermarkt.py`) | Season-start **squad market value** | Quality / cumulative-spend proxy, fixed before kickoff |

All external features are built by `scripts/pipeline/build_external_features.py` and every
rolling stat is `shift(1)`-ed, so a match only ever sees strictly-prior data.

> **Headline finding (documented below):** once a subtle leakage in Understat's own
> post-hoc "forecast" is removed, legitimate pre-match features land at **~0.44–0.45
> balanced accuracy — essentially the bookmaker's own ceiling.** The market already
> prices in form, xG and squad value, so they add little on top of it. 65% balanced
> accuracy on 3-class H/D/A is above what the betting market itself achieves.

## 1. Imports & data

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, confusion_matrix)
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

BASE = pd.read_csv("../data/processed/epl_features.csv", parse_dates=["Date"])
EXT = pd.read_csv("../data/processed/external/epl_external_features.csv")

# Drop Understat's post-hoc forecast columns (leakage — see section 5).
EXT = EXT.drop(columns=[col for col in EXT.columns if col.startswith("understat_")])

df = BASE.merge(EXT.drop(columns=["Date"]), on=["Season", "HomeTeam", "AwayTeam"], how="left")
print("rows:", len(df), "| external cols merged:", EXT.shape[1] - 4)
df[["Season","HomeTeam","AwayTeam","elo_diff","value_diff_m","diff_xg_xg_diff_last_10"]].head()

## 2. Chronological split & target

- Target `target_result_encoded`: `0 = Home`, `1 = Draw`, `2 = Away`.
- Train ≤ 2023_2024, validation = 2024_2025, test = 2025_2026 (reported once).

In [ ]:
y = df["target_result_encoded"]
train = df["Season"] <= "2023_2024"
valid = df["Season"] == "2024_2025"
test  = df["Season"] == "2025_2026"
trainval = df["Season"] <= "2024_2025"

dist = (df["target_result"].value_counts(normalize=True)
        .rename({"H":"Home","D":"Draw","A":"Away"}).mul(100).round(1))
print("Outcome distribution (%):"); print(dist)

## 3. Feature groups

Small, curated groups — deliberately avoiding the 300-feature dump that *underperforms* the market (curse of dimensionality).

In [ ]:
market = ["market_home_norm_prob","market_draw_norm_prob","market_away_norm_prob","market_overround"]
elo    = ["elo_diff","elo_expected_home"]
xg     = [c for c in df.columns if c.startswith("diff_xg_")]
value  = ["value_diff_m","value_rank_diff","value_log_ratio"]
form   = ["diff_season_points_per_match","diff_overall_goal_diff_last_5",
         "diff_overall_points_last_5","diff_table_position",
         "diff_venue_season_goal_diff_per_match","season_progress"]

ALL = market + elo + xg + value + form
print({"market":len(market),"elo":len(elo),"xg":len(xg),"value":len(value),"form":len(form),"ALL":len(ALL)})

## 4. Evaluation helpers & feature-set sweep

In [ ]:
def make_lr():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(max_iter=5000, class_weight="balanced", C=0.5))

def make_gbm():
    return make_pipeline(SimpleImputer(strategy="median"),
        LGBMClassifier(n_estimators=500, learning_rate=0.015, num_leaves=15,
                       min_child_samples=50, subsample=0.85, colsample_bytree=0.7,
                       reg_lambda=2.0, class_weight="balanced", verbosity=-1, n_jobs=-1))

def evaluate(feats, make_model, fit_mask=trainval):
    model = make_model()
    model.fit(df.loc[fit_mask, feats], y[fit_mask])
    yp = model.predict(df.loc[test, feats])
    return dict(acc=accuracy_score(y[test], yp),
                bal=balanced_accuracy_score(y[test], yp),
                f1=f1_score(y[test], yp, average="macro")), model

subsets = {"market":market, "market+elo":market+elo, "market+elo+value":market+elo+value,
           "market+elo+xg":market+elo+xg, "market+elo+xg+value":market+elo+xg+value,
           "ALL":ALL}
rows = []
for name, fs in subsets.items():
    m, _ = evaluate(fs, make_lr)
    rows.append({"feature_set":name, "n":len(fs), **{k:round(v,3) for k,v in m.items()}})
sweep = pd.DataFrame(rows); sweep

## 5. ⚠️ Leakage lesson: Understat's `forecast` is not a prediction

Understat publishes a per-match `forecast` (win/draw/loss probability). It is tempting
to use as a feature — but it is a **Poisson simulation of that same match's realised
xG**, i.e. a *retrodiction*. Its correlation with the match's own xG margin is ~0.96,
so it silently encodes the outcome. Included, it inflates balanced accuracy to ~0.52;
it is **excluded everywhere in this notebook.** This is the single most important
methodological catch in the project.

In [ ]:
raw = []
import glob
cols = ["date","home","away","xg_h","xg_a","g_h","g_a","fc_w","fc_d","fc_l"]
for f in sorted(glob.glob("../data/raw/external/understat/EPL_*.tsv")):
    raw.append(pd.read_csv(f, sep="\t", header=None, names=cols))
us = pd.concat(raw, ignore_index=True)
print("corr(Understat home-win forecast, SAME match xG margin):",
      round(us["fc_w"].corr(us["xg_h"] - us["xg_a"]), 3), "-> leakage, dropped")

## 6. Final honest evaluation

In [ ]:
(m_lr, _)  = evaluate(ALL, make_lr)
(m_gbm, gbm) = evaluate(ALL, make_gbm)
print(f"LogReg   acc={m_lr['acc']:.3f}  balanced={m_lr['bal']:.3f}  macroF1={m_lr['f1']:.3f}")
print(f"LightGBM acc={m_gbm['acc']:.3f}  balanced={m_gbm['bal']:.3f}  macroF1={m_gbm['f1']:.3f}")

mp = df.loc[test, ["market_home_norm_prob","market_draw_norm_prob","market_away_norm_prob"]].values
print(f"\nReference — bookmaker odds argmax: balanced={balanced_accuracy_score(y[test], mp.argmax(1)):.3f}")

## 7. Where high accuracy *is* reachable

3-class balanced accuracy is capped near the market (~0.44). But 65%+ **is** attainable
if the problem is reframed — useful context for the capstone narrative:

In [ ]:
model = make_gbm(); model.fit(df.loc[trainval, ALL], y[trainval])
P = model.predict_proba(df.loc[test, ALL]); yt = y[test].values; yp = P.argmax(1)

yb = (yt == 0).astype(int); pb = (P[:,0] > 0.5).astype(int)
print(f"Binary (Home vs not) accuracy: {accuracy_score(yb, pb):.3f}  (base rate {1-yb.mean():.3f})")
conf = P.max(1)
for thr in [0.5, 0.6, 0.7]:
    mk = conf >= thr
    print(f"Confident subset maxprob>={thr}: {mk.sum():3d} games, accuracy={accuracy_score(yt[mk], yp[mk]):.3f}")

## 8. Confusion matrix (best model, 3-class test season)

In [ ]:
cm = confusion_matrix(yt, yp, labels=[0,1,2])
plt.figure(figsize=(5.5,4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Home","Draw","Away"], yticklabels=["Home","Draw","Away"])
plt.title("LightGBM + external features — 2025_2026"); plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout(); plt.show()

## 9. Takeaways

1. **Elo, xG-form and squad value are real, correctly-built pre-match signals — but they
   barely beat the market**, because efficient betting odds already aggregate them.
2. **The honest 3-class balanced-accuracy ceiling is ~0.44–0.45**, matching the bookmaker.
   The only thing that reached ~0.52 was leakage (Understat forecast).
3. **Fewer features win.** The curated ~20–29 feature set matches or beats a 300-feature
   dump, confirming the curse-of-dimensionality concern.
4. **65% is reachable only by changing the question** — binary Home-vs-not (~0.63) or
   scoring only high-confidence games (0.73 on maxprob≥0.7).

**Re-run the data pipeline:**
```
python scripts/scrape/fetch_clubelo.py
python scripts/scrape/fetch_transfermarkt.py
# Understat: run scripts/scrape/_understat_sink.py, then load each season page in a
# browser and POST datesData to it (Cloudflare blocks headless fetch).
python scripts/pipeline/build_external_features.py
```